# 03 - From-scratch DPO on UltraFeedback

Accelerator: GPU T4 x2, Internet on. Input: the output of notebook 01 (Add Input -> Your Work -> Notebooks).
Policy on cuda:0, frozen reference on cuda:1, both initialised from the SFT `best` checkpoint.
Outputs: `outputs/dpo_ultrafeedback/{best,final,metrics.jsonl,config.json,data_summary.json}`.

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO = "small-model-alignment"
GITHUB_USER = "ahmedamirkol01-art"
os.chdir("/kaggle/working")
if not os.path.exists(REPO):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    result = subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"], capture_output=True, text=True)
    print("cloned" if result.returncode == 0 else result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GITHUB_USER}/{REPO}.git"])
os.chdir(f"/kaggle/working/{REPO}")
os.environ["PYTHONPATH"] = "."
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!git log --oneline -3

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("visible GPUs:", torch.cuda.device_count())

In [ ]:
!python -m pytest -q tests/

In [ ]:
import glob

SFT_BEST = glob.glob("/kaggle/input/**/outputs/sft/best", recursive=True)[0]
print(SFT_BEST)

Pilot run: step 0 must show reward accuracy 0 and margin 0, since policy and reference are identical.

In [ ]:
!python scripts/train_dpo.py --dataset ultrafeedback --sft_path {SFT_BEST} --output_dir outputs/dpo_ultrafeedback_pilot --train_size 64 --val_size 16 --grad_accum 8 --eval_every 4
!head -2 outputs/dpo_ultrafeedback_pilot/metrics.jsonl
!rm -rf outputs/dpo_ultrafeedback_pilot

Full run with the guide configuration.

In [ ]:
!python scripts/train_dpo.py --dataset ultrafeedback --sft_path {SFT_BEST} --output_dir outputs/dpo_ultrafeedback

In [ ]:
!python scripts/plot_metrics.py --out_dir outputs/plots
!cd /kaggle/working && zip -r -q dpo_ultrafeedback_run_outputs.zip small-model-alignment/outputs -x "*.safetensors"
!ls -lh outputs/dpo_ultrafeedback outputs/dpo_ultrafeedback/best